# Gesture Synth Experiments

Use this notebook to test oscillator waveforms and debug webcam hand tracking without running the full app.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import numpy as np

from src.config import AppConfig, SynthConfig
from src.synth import Synthesizer

config = AppConfig.load(PROJECT_ROOT / 'config.json')
config

## Oscillator generation

In [ ]:
for waveform in ['sine', 'square', 'sawtooth', 'triangle']:
    synth = Synthesizer(SynthConfig(sample_rate=44100, waveform=waveform, attack=0.001))
    synth.note_on('C4', 261.63)
    audio = synth.render(1200)
    plt.figure(figsize=(10, 2.5))
    plt.plot(audio[:400])
    plt.title(waveform)
    plt.ylim(-0.35, 0.35)
    plt.grid(True, alpha=0.25)
    plt.show()

## Webcam hand landmarks

Run this cell locally with a webcam. Press `q` in the OpenCV window to stop.

In [ ]:
import cv2

from src.camera import Camera
from src.gesture_detector import GestureStabilizer, count_extended_fingers, supported_gesture
from src.hand_tracker import HandTracker
from src.ui import FPSCounter, draw_overlay

camera = Camera(config.camera)
tracker = HandTracker(config.gesture)
stabilizer = GestureStabilizer(config.gesture.stable_frames)
fps = FPSCounter()

try:
    while True:
        frame = camera.read()
        if config.gesture.mirror_camera:
            frame = cv2.flip(frame, 1)
        detection, results = tracker.process(frame)
        raw = None
        if detection:
            raw = count_extended_fingers(detection.landmarks.landmark, detection.handedness, mirrored=config.gesture.mirror_camera)
        state = stabilizer.update(supported_gesture(raw, config.gesture_notes.keys()))
        note = config.gesture_notes.get(state.stable_fingers)
        tracker.draw(frame, results)
        draw_overlay(frame, state, note, config.synth.waveform, fps.update())
        cv2.imshow('Gesture Debug', frame)
        if cv2.waitKey(1) & 0xFF in (ord('q'), 27):
            break
finally:
    tracker.close()
    camera.close()
    cv2.destroyAllWindows()